In [11]:
import os
from pathlib import Path
import pandas as pd
from qiskit import QuantumCircuit

def qasm_dataset_report(
    qasm_dir: os.PathLike,
    dataset_name: str,
) -> pd.DataFrame:
    """
    Load every *.qasm file in `qasm_dir`, compute basic statistics,
    and return a one-row pandas DataFrame.

    Columns
    -------
    dataset   : name of the dataset (optional)
    n_circuits: number of circuits scanned
    depth_min : minimum circuit depth
    depth_max : maximum circuit depth
    ops_min   : minimum gate count across circuits
    ops_max   : maximum gate count across circuits
    qubits_min: minimum number of qubits
    qubits_max: maximum number of qubits
    """
    qasm_dir = Path(qasm_dir)
    if dataset_name is None:
        dataset_name = qasm_dir.name

    depths, ops_counts, qubits_counts = [], [], []

    for qasm_file in qasm_dir.glob("*.qasm"):
        try:
            circ = QuantumCircuit.from_qasm_file(qasm_file)
        except Exception as e:
            # Skip broken files; or raise if you prefer
            print(f"Skipping {qasm_file}: {e}")
            continue

        depths.append(circ.depth())
        ops_counts.append(sum(circ.count_ops().values()))
        qubits_counts.append(circ.num_qubits)

    if not depths:          # empty folder
        return pd.DataFrame([{
            "dataset": dataset_name,
            "n_circuits": 0,
            "depth_min": None, "depth_max": None,
            "ops_min": None, "ops_max": None,
            "qubits_min": None, "qubits_max": None
        }])

    return pd.DataFrame([{
        "dataset": dataset_name,
        "n_circuits": len(depths),
        "depth_min": min(depths),
        "depth_max": max(depths),
        "ops_min": min(ops_counts),
        "ops_max": max(ops_counts),
        "qubits_min": min(qubits_counts),
        "qubits_max": max(qubits_counts)
    }])

In [12]:
qasm_dataset_report(Path('./data/nam_circs'), "NAM Circuits")

,dataset,n_circuits,depth_min,depth_max,ops_min,ops_max,qubits_min,qubits_max
0,NAM Circuits,19,33,339,45,495,5,30
